# 01 · Importación de datos

**Objetivo:** cargar los préstamos de Lending Club, quedarse solo con los que sirven para modelizar el riesgo de crédito (**PD**, **EAD** y **LGD**) y dividirlos en un conjunto de **entrenamiento** (70 %) y otro de **validación** (30 %), reservado para evaluar los modelos al final.

Todas las decisiones se apoyan en datos que se calculan en este mismo notebook. Cada una aparece en un recuadro **Decisión Dx** con *qué* se hace, *por qué* y *qué alternativas se descartaron*.

<a id="s0"></a>

## Registro de decisiones

| # | Decisión | Motivo principal |
|---|---|---|
| D1 | Usar el dataset público de Lending Club (Kaggle, CC0) sin versionarlo en Git | Licencia libre, pero 1,7 GB no caben en un repositorio |
| D2 | Leer el CSV por bloques y solo con las columnas necesarias | El archivo completo no cabe en la memoria de un equipo normal |
| D3 | Eliminar las filas de totales | No son préstamos: son restos de unir los CSV trimestrales |
| D4 | Quedarse solo con préstamos terminados | Un préstamo en curso todavía no tiene resultado |
| D5 | Usar solo cohortes con ≥ 95 % de préstamos terminados | Evita el sesgo de las cohortes recientes |
| D6 | Seleccionar columnas según su papel y separar las que causan *leakage* | Solo pueden entrar variables conocidas al conceder el préstamo |
| D7 | Empezar en 2012 | Antes faltan variables del modelo (nulos estructurales) |
| D8 | Solo solicitudes individuales | En las conjuntas, ingresos y DTI significan otra cosa |
| D9 | Eliminar columnas con > 80 % de nulos | No aportan información fiable |
| D10 | Renombrar las columnas al español | Coherencia con el resto del proyecto (notebooks 02–08 y API) |
| D11 | No limpiar valores aquí (solo convertir fechas) | La calidad de datos se trata en el notebook 02 |
| D12 | Partición 70/30 estratificada con semilla fija | Reproducible y con la misma tasa de impago en ambos conjuntos |

## Índice

- [Registro de decisiones](#s0)
- [1 · Importar librerías y configuración](#s1)
- [2 · D1 · Fuente de datos](#s2)
- [3 · D2 · Lectura por bloques](#s3)
  - [3.1 · Primera lectura: columnas mínimas](#s3-1)
- [4 · D3 · Filas de totales](#s4)
- [5 · D4 · Estado del préstamo](#s5)
- [6 · D5 · Cohortes](#s6)
- [7 · D6 · Selección de columnas](#s7)
  - [7.1 · Clasificación de las 151 columnas](#s7-1)
  - [7.2 · Diccionario de las variables seleccionadas](#s7-2)
  - [7.3 · Segunda lectura: columnas seleccionadas](#s7-3)
- [8 · D7 · Año de inicio](#s8)
- [9 · D8 · Tipo de solicitud](#s9)
- [10 · D9 · Columnas casi vacías](#s10)
- [11 · D10 · Nombres en español](#s11)
- [12 · D11 · Tipos de datos](#s12)
- [13 · D12 · Partición entrenamiento / validación](#s13)
- [14 · Guardar los conjuntos](#s14)
- [15 · Resumen: embudo de filas](#s15)
- [16 · Próximos pasos](#s16)

> Cada sección empieza con una **franja azul**. Debajo de cada resultado hay una explicación de **qué se hace** y **qué se observa**.

<a id="s1"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">1 · IMPORTAR LIBRERÍAS Y CONFIGURACIÓN</span>

</div>

In [1]:
# --- Librerías ------------------------------------------------------------
from pathlib import Path                               # rutas de archivos independientes del sistema operativo
import numpy as np                                     # cálculo numérico
import pandas as pd                                    # manejo de tablas (DataFrames)
from sklearn.model_selection import train_test_split   # partición entrenamiento / validación

# --- Opciones de visualización --------------------------------------------
pd.set_option('display.float_format', lambda x: '%.3f' % x)  # sin notación científica
pd.set_option('display.max_columns', 60)                     # mostrar todas las columnas de una tabla
pd.set_option('display.max_rows', 200)                       # mostrar tablas largas completas (p. ej. las 151 columnas)
np.set_printoptions(suppress=True)

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


**Qué se hace:** se importan las librerías del notebook y se configura cómo se muestran las tablas (sin notación científica y con todas las columnas y filas visibles).

**Qué se observa:** el mensaje confirma que todas las librerías están instaladas en el entorno.

In [2]:
# --- Rutas (relativas a la carpeta de este notebook: 03_notebooks/es) ------
RUTA_DATOS = Path('../../02_datos')
RUTA_ORIGINAL = RUTA_DATOS / '00_Originales' / 'accepted_2007_to_2018Q4.csv'
RUTA_ENTRENAMIENTO = RUTA_DATOS / '03_Entrenamiento' / 'train.pkl'
RUTA_VALIDACION = RUTA_DATOS / '02_Validacion' / 'validacion.pkl'

# --- Parámetros de las decisiones (cada uno se justifica más abajo) --------
TAM_BLOQUE = 200_000        # D2: filas que se leen en cada bloque
UMBRAL_TERMINADOS = 0.95    # D5: proporción mínima de préstamos terminados para aceptar una cohorte
ANIO_INICIO = 2012          # D7: primer año de concesión que se conserva
UMBRAL_NULOS = 0.80         # D9: proporción máxima de nulos que se tolera en una columna
PROP_VALIDACION = 0.30      # D12: proporción reservada para validación
SEMILLA = 42                # D12: semilla fija para que la partición sea reproducible

# Diccionario donde se anota cuántas filas quedan tras cada paso (embudo del final)
embudo = {}

**Qué se hace:** se definen las rutas de los archivos y los parámetros de todas las decisiones del notebook, en un único sitio para poder cambiarlos fácilmente. Esta celda no produce ningún resultado.

<a id="s2"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">2 · D1 · FUENTE DE DATOS</span>

</div>

> **Decisión D1 · Usar el dataset público *All Lending Club loan data* (Kaggle, licencia CC0) y no subirlo a Git.**
> - **Qué:** el archivo `accepted_2007_to_2018Q4.csv`, con los préstamos **concedidos** entre 2007 y 2018. El archivo de solicitudes rechazadas no se usa porque no tiene resultado de pago y, por tanto, no sirve para PD, EAD ni LGD.
> - **Por qué:** son datos reales de una plataforma de préstamos entre particulares con licencia de dominio público (CC0), así que se pueden usar y compartir libremente.
> - **Por qué no se sube a Git:** ocupa 1,7 GB. Está excluido en `.gitignore`.
> - **Cómo reproducirlo:** descargar el dataset de Kaggle (autor *wordsforthewise*), descomprimir el archivo de aceptados y guardarlo en `02_datos/00_Originales/`.

In [3]:
# Comprobar que el archivo existe antes de seguir (si no, explicar cómo conseguirlo)
if not RUTA_ORIGINAL.exists():
    raise FileNotFoundError(
        f"No se encuentra {RUTA_ORIGINAL}. Descarga 'All Lending Club loan data' de Kaggle "
        "y guarda accepted_2007_to_2018Q4.csv en 02_datos/00_Originales/"
    )

# Tamaño del archivo en GB
print(f"Tamaño del archivo: {RUTA_ORIGINAL.stat().st_size / 1e9:.2f} GB")

# Leer solo la cabecera (nrows=0) para conocer las columnas sin cargar ningún dato
cabecera = pd.read_csv(RUTA_ORIGINAL, nrows=0)
print(f"Número de columnas: {cabecera.shape[1]}")

Tamaño del archivo: 1.68 GB
Número de columnas: 151


**Qué se hace:** se comprueba que existe el archivo original y se lee **solo la cabecera** (`nrows=0`), sin cargar ningún dato, para conocer sus columnas.

**Qué se observa:** el archivo ocupa **1,68 GB** y tiene **151 columnas**. Por su tamaño no conviene cargarlo entero (decisión D2).

<a id="s3"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">3 · D2 · LECTURA POR BLOQUES</span>

</div>

> **Decisión D2 · Leer el CSV por bloques de 200.000 filas, cargando solo las columnas necesarias y filtrando cada bloque antes de guardarlo.**
> - **Por qué:** el archivo tiene 2,26 millones de filas y 151 columnas. Cargado entero en pandas ocupa varios GB de memoria RAM (en un equipo con 4 GB ni siquiera caben 40 columnas). Leyendo por bloques, en memoria solo se acumulan las filas que pasan el filtro.
> - **Cómo:** la función `leer_por_bloques` se usa dos veces. Primero se leen 4 columnas de todas las filas para decidir **qué filas** sirven (D3–D5). Después se leen las columnas elegidas en D6, **solo** de esas filas.
> - **Alternativas descartadas:** cargarlo todo de golpe (falla por memoria) o convertirlo antes a Parquet (requiere librerías extra; puede plantearse más adelante si la lectura resulta lenta).

In [4]:
def leer_por_bloques(columnas, filtro=None, tam_bloque=TAM_BLOQUE):
    '''Lee el CSV original por bloques y devuelve un único DataFrame.

    columnas : lista de columnas a leer (el resto ni siquiera se carga)
    filtro   : función opcional que recibe un bloque y devuelve una máscara booleana
               con las filas que se quieren conservar
    '''
    bloques = []                     # aquí se acumulan los trozos ya filtrados
    lector = pd.read_csv(
        RUTA_ORIGINAL,
        usecols=columnas,            # solo las columnas pedidas
        chunksize=tam_bloque,        # devuelve un iterador de bloques en vez de un único DataFrame
        dtype={'id': str},           # 'id' como texto: las filas de totales tienen texto en esa columna (D3)
        low_memory=False,            # deducir el tipo de cada columna usando el bloque completo
    )
    for bloque in lector:
        if filtro is not None:
            bloque = bloque[filtro(bloque)]   # conservar solo las filas que cumplen el filtro
        bloques.append(bloque)
    return pd.concat(bloques)        # unir todos los bloques (se conserva el índice original)

**Qué se hace:** se define la función que lee el CSV por bloques (decisión D2). Esta celda no produce ningún resultado: la función se usa en la primera y en la segunda lectura.

<a id="s3-1"></a>

### 3.1 · Primera lectura: las columnas mínimas para decidir qué filas sirven

In [5]:
# Primera lectura: identificador, fecha de concesión, plazo y estado del préstamo
base = leer_por_bloques(['id', 'issue_d', 'term', 'loan_status'])
embudo['1. Archivo original'] = len(base)
print(f"Filas leídas: {len(base):,}")

Filas leídas: 2,260,701


**Qué se hace:** primera lectura por bloques: solo 4 columnas (identificador, fecha de concesión, plazo y estado), pero de **todas** las filas.

**Qué se observa:** el archivo tiene **2.260.701 filas**. Con estas 4 columnas se decide qué filas sirven (D3–D5) antes de leer el resto.

<a id="s4"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">4 · D3 · FILAS DE TOTALES</span>

</div>

In [6]:
# Filas sin estado del préstamo: ¿qué contienen?
filas_sin_estado = base[base['loan_status'].isna()]
print(f"Filas sin estado: {len(filas_sin_estado)}")
filas_sin_estado.head(6)

Filas sin estado: 33


,id,term,issue_d,loan_status
421095,Total amount funded in policy code 1: 6417608175,NaN,NaN,NaN
421096,Total amount funded in policy code 2: 1944088810,NaN,NaN,NaN
528961,Total amount funded in policy code 1: 1741781700,NaN,NaN,NaN
528962,Total amount funded in policy code 2: 564202131,NaN,NaN,NaN
651664,Total amount funded in policy code 1: 1791201400,NaN,NaN,NaN
651665,Total amount funded in policy code 2: 651669342,NaN,NaN,NaN


**Qué se hace:** se buscan las filas sin estado del préstamo y se muestran las primeras.

**Qué se observa:** hay **33 filas** sin estado. No son préstamos: en la columna `id` contienen textos como *"Total amount funded in policy code 1: …"* y el resto de columnas está vacío.

> **Decisión D3 · Eliminar las filas sin `loan_status`.**
> - **Por qué:** no son préstamos, sino líneas de totales (*"Total amount funded in policy code…"*) que quedaron al unir los CSV trimestrales que publicaba Lending Club.

In [7]:
# Eliminar las filas de totales
base = base.dropna(subset=['loan_status'])

# Sin esas filas, el identificador ya se puede convertir a número entero
base['id'] = base['id'].astype('int64')

embudo['2. Sin filas de totales (D3)'] = len(base)
print(f"Filas restantes: {len(base):,}")

Filas restantes: 2,260,668


**Qué se hace:** se eliminan las 33 filas de totales y se convierte el identificador a número entero (ya solo contiene números).

**Qué se observa:** quedan **2.260.668 filas**.

<a id="s5"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">5 · D4 · ESTADO DEL PRÉSTAMO</span>

</div>

**Variable analizada: `loan_status` (en español, `estado`)** · estado del préstamo en la fecha de corte del dataset (finales de 2018):
- `Fully Paid`: devuelto por completo.
- `Charged Off`: dado por perdido (**impago**): Lending Club deja de esperar los pagos.
- `Current`: en curso y al corriente de pago.
- `In Grace Period`: retraso de hasta 15 días (periodo de gracia).
- `Late (16-30 days)` / `Late (31-120 days)`: pago atrasado ese número de días.
- `Default`: más de 120 días de retraso, todavía sin darse por perdido.
- `Does not meet the credit policy. Status: …`: préstamos antiguos concedidos con criterios que ya no cumplen la política de crédito vigente; después de los dos puntos aparece su estado final.

In [8]:
# Estados que indican que el préstamo ya ha terminado (resultado conocido)
ESTADOS_TERMINADOS = [
    'Fully Paid',
    'Charged Off',
    'Does not meet the credit policy. Status:Fully Paid',
    'Does not meet the credit policy. Status:Charged Off',
]
# Estados terminados que cuentan como impago (mismo criterio que el notebook 04)
ESTADOS_IMPAGO = ['Charged Off', 'Does not meet the credit policy. Status:Charged Off']

# Tabla de estados: número de préstamos, porcentaje y si el resultado es conocido
tabla_estados = base['loan_status'].value_counts().to_frame('n_prestamos')
tabla_estados['%'] = tabla_estados['n_prestamos'] / len(base) * 100
tabla_estados['terminado'] = tabla_estados.index.isin(ESTADOS_TERMINADOS)
tabla_estados['impago'] = tabla_estados.index.isin(ESTADOS_IMPAGO)
tabla_estados

,n_prestamos,%,terminado,impago
loan_status,,,,
Fully Paid,1076751,47.630,True,False
Current,878317,38.852,False,False
Charged Off,268559,11.880,True,True
Late (31-120 days),21467,0.950,False,False
In Grace Period,8436,0.373,False,False
Late (16-30 days),4349,0.192,False,False
Does not meet the credit policy. Status:Fully Paid,1988,0.088,True,False
Does not meet the credit policy. Status:Charged Off,761,0.034,True,True
Default,40,0.002,False,False


**Qué se hace:** tabla con el número y el % de préstamos de cada estado. Las columnas `terminado` e `impago` indican si el estado es definitivo y si cuenta como impago.

**Qué se observa:** **Fully Paid** (47,6 %) y **Charged Off** (11,9 %) son los terminados mayoritarios. **Current** (38,9 %) son préstamos todavía en curso, sin resultado conocido. Los retrasos (`Late`, `In Grace Period`, `Default`) suman en torno al 1,5 %.

> **Decisión D4 · Conservar solo los préstamos terminados**: `Fully Paid`, `Charged Off` y sus variantes *"Does not meet the credit policy"*.
> - **Por qué:** la PD estima si un préstamo **acabará** en impago. Un préstamo `Current`, `Late` o `In Grace Period` todavía puede terminar de cualquier forma. Si se etiqueta como "no impago", el modelo aprende con etiquetas erróneas. Con EAD y LGD pasa lo mismo: necesitan el importe final amortizado y recuperado.
> - `Default` tampoco está cerrado: es un retraso de más de 120 días que aún no se ha dado por perdido.
> - **Alternativa descartada:** contar `Late (31-120 days)` y `Default` como impago. Es habitual en banca (definición de 90 días de retraso), pero mezclaría préstamos cuyo resultado final no conocemos.

In [9]:
# Marcar cada préstamo: ¿ha terminado? ¿ha terminado en impago?
base['terminado'] = base['loan_status'].isin(ESTADOS_TERMINADOS)
base['impago'] = base['loan_status'].isin(ESTADOS_IMPAGO)

embudo['3. Solo préstamos terminados (D4)'] = int(base['terminado'].sum())
print(f"Préstamos terminados: {base['terminado'].sum():,}")

Préstamos terminados: 1,348,059


**Qué se hace:** se marca cada préstamo según si ha terminado y si ha terminado en impago.

**Qué se observa:** hay **1.348.059 préstamos terminados** (el 59,6 % del archivo).

<a id="s6"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">6 · D5 · COHORTES</span>

</div>

Una **cohorte** es el grupo de préstamos concedidos el mismo año y con el mismo plazo (36 o 60 meses). El dataset llega hasta finales de 2018, así que muchas cohortes recientes todavía no han terminado.

**Variables analizadas:**
- `issue_d` (en español, `fecha_concesion`): mes y año en que se concedió el préstamo (formato `Dec-2015`).
- `term` (en español, `num_cuotas`): plazo del préstamo: 36 o 60 cuotas mensuales.

In [10]:
# Año de concesión ('issue_d' tiene el formato 'Dec-2015')
base['anio'] = pd.to_datetime(base['issue_d'], format='%b-%Y').dt.year

# Plazo sin el espacio inicial (' 36 months' -> '36 months')
base['plazo'] = base['term'].str.strip()

# Por cohorte (plazo, año): nº de préstamos y proporción de terminados
cohortes = base.groupby(['plazo', 'anio']).agg(
    n_prestamos=('id', 'size'),
    pct_terminados=('terminado', 'mean'),
)
# Tasa de impago calculada SOLO entre los préstamos terminados de cada cohorte
cohortes['tasa_impago_terminados'] = base[base['terminado']].groupby(['plazo', 'anio'])['impago'].mean()

# Una cohorte es válida si casi todos sus préstamos han terminado
cohortes['valida'] = cohortes['pct_terminados'] >= UMBRAL_TERMINADOS
cohortes

n_prestamos  pct_terminados  tasa_impago_terminados  valida
plazo     anio                                                             
36 months 2007          603           1.000                   0.262    True
          2008         2393           1.000                   0.207    True
          2009         5281           1.000                   0.137    True
          2010         9156           1.000                   0.109    True
          2011        14101           1.000                   0.106    True
          2012        43470           1.000                   0.136    True
          2013       100422           1.000                   0.123    True
          2014       162570           1.000                   0.137    True
          2015       283173           0.999                   0.149    True
          2016       323495           0.718                   0.198   False
          2017       320419           0.401                   0.200   False
          2018       344671           0.120                   0.132   False
60 months 2010         3381           1.000                   0.224    True
          2011         7620           1.000                   0.236    True
          2012         9897           1.000                   0.277    True
          2013        34392           1.000                   0.251    True
          2014        73059           0.829                   0.311   False
          2015       137922           0.671                   0.364   False
          2016       110912           0.548                   0.364   False
          2017       123160           0.331                   0.329   False
          2018       150571           0.100                   0.226   False

**Qué se hace:** para cada **cohorte** (plazo × año de concesión) se calcula el nº de préstamos, la proporción de terminados (`pct_terminados`, de 0 a 1), la tasa de impago **solo entre los terminados** (`tasa_impago_terminados`) y si la cohorte es válida (`valida`: al menos un 95 % de terminados).

**Qué se observa:** a 36 meses, las cohortes están completas hasta 2015 (≥ 99,9 %) y caen después (72 % en 2016). A 60 meses, solo hasta 2013 (83 % en 2014). En las cohortes incompletas a 60 meses, la tasa de impago **aparente** se dispara (31–36 %) porque los impagos llegan antes que los pagos completos.

> **Decisión D5 · Usar solo las cohortes con al menos un 95 % de préstamos terminados.**
> - **Por qué:** en una cohorte reciente, los préstamos que ya han terminado no son representativos. Un impago suele llegar antes que el último pago de un préstamo que se paga con normalidad, así que entre los terminados hay "demasiados" impagos (**sesgo de selección**). Se ve en la tabla: en los préstamos a 60 meses de 2015 solo ha terminado un 67 % y su tasa de impago aparente es del 36 %, frente al 25 % de la cohorte de 2013, que ya está completa.
> - **Resultado:** se conservan los préstamos a 36 meses hasta 2015 y los de 60 meses hasta 2013 (columna `valida`).
> - **Alternativas descartadas:** usar todos los terminados (sesgo de selección) o modelos de supervivencia, que tienen en cuenta los préstamos no terminados (más complejos y fuera del alcance de este proyecto).

In [11]:
# Lista de cohortes válidas como pares (plazo, año)
cohortes_validas = cohortes[cohortes['valida']].index.tolist()
print("Cohortes válidas:")
for plazo, anio in cohortes_validas:
    print(f"  {plazo} · {anio}")

# ¿Pertenece cada préstamo a una cohorte válida?
base['en_cohorte_valida'] = pd.MultiIndex.from_arrays([base['plazo'], base['anio']]).isin(cohortes_validas)

# Identificadores de los préstamos que superan D3, D4 y D5
ids_seleccionados = base.loc[base['terminado'] & base['en_cohorte_valida'], 'id']
embudo['4. En cohortes con ≥ 95 % terminados (D5)'] = len(ids_seleccionados)
print(f"\nPréstamos seleccionados: {len(ids_seleccionados):,}")

# Liberar memoria: la tabla 'base' ya no hace falta (solo los identificadores seleccionados)
del base

Cohortes válidas:
  36 months · 2007
  36 months · 2008
  36 months · 2009
  36 months · 2010
  36 months · 2011
  36 months · 2012
  36 months · 2013
  36 months · 2014
  36 months · 2015
  60 months · 2010
  60 months · 2011
  60 months · 2012
  60 months · 2013

Préstamos seleccionados: 676,302


**Qué se hace:** se listan las cohortes válidas y se guardan los identificadores de los préstamos que superan D3, D4 y D5. Después se libera la memoria de la tabla `base`.

**Qué se observa:** hay **13 cohortes válidas** (36 meses de 2007 a 2015 y 60 meses de 2010 a 2013) con **676.302 préstamos**.

<a id="s7"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">7 · D6 · SELECCIÓN DE COLUMNAS</span>

</div>

El archivo tiene 151 columnas. Cada una se clasifica según su **papel**:

- **Identificador** del préstamo.
- **Originación**: lo que se sabe **al solicitar** el préstamo. Son las únicas que pueden ser variables del modelo.
- **Objetivo**: lo que ocurre **después** de conceder el préstamo. Solo sirven para construir lo que se quiere predecir (PD, EAD, LGD).
- **Auxiliar**: solo para filtrar o partir los datos.
- **Descartada**: el resto, con su motivo.

> **¿Qué es la fuga de información (*leakage*)?** Entrenar con datos que no se conocen en el momento de predecir. Por ejemplo, `total_pymnt` (total pagado por el cliente) delata si el préstamo se pagó. Un modelo que la use tendría un AUC altísimo en el notebook y sería inútil en la demo, donde se evalúa a un cliente **antes** de concederle el préstamo.

In [12]:
# Columnas que se conservan: nombre en Lending Club -> (nombre en español, grupo)
COLUMNAS = {
    # --- Identificadores -------------------------------------------------
    'id':                       ('id_cliente',                  'Identificador'),
    'member_id':                ('id_prestamo',                 'Identificador'),
    # --- Originación: datos de la solicitud y del préstamo (14) ----------
    'emp_title':                ('empleo',                      'Originación'),
    'emp_length':               ('antigüedad_empleo',           'Originación'),
    'annual_inc':               ('ingresos',                    'Originación'),
    'verification_status':      ('ingresos_verificados',        'Originación'),
    'home_ownership':           ('vivienda',                    'Originación'),
    'dti':                      ('dti',                         'Originación'),
    'desc':                     ('descripcion',                 'Originación'),
    'purpose':                  ('finalidad',                   'Originación'),
    'funded_amnt':              ('principal',                   'Originación'),
    'term':                     ('num_cuotas',                  'Originación'),
    'int_rate':                 ('tipo_interes',                'Originación'),
    'installment':              ('imp_cuota',                   'Originación'),
    'grade':                    ('rating',                      'Originación'),
    'sub_grade':                ('subrating',                   'Originación'),
    # --- Originación: historial del buró de crédito (16) -----------------
    'fico_range_low':           ('fico_min',                    'Originación'),
    'fico_range_high':          ('fico_max',                    'Originación'),
    'earliest_cr_line':         ('fecha_primer_credito',        'Originación'),
    'open_acc':                 ('num_lineas_credito',          'Originación'),
    'total_acc':                ('num_lineas_credito_total',    'Originación'),
    'mort_acc':                 ('num_hipotecas',               'Originación'),
    'revol_bal':                ('saldo_revolving',             'Originación'),
    'revol_util':               ('porc_uso_revolving',          'Originación'),
    'bc_util':                  ('porc_uso_tarjetas',           'Originación'),
    'percent_bc_gt_75':         ('porc_tarjetas_75p',           'Originación'),
    'inq_last_6mths':           ('num_consultas_6meses',        'Originación'),
    'delinq_2yrs':              ('num_retrasos_2anios',         'Originación'),
    'mths_since_last_delinq':   ('num_meses_desde_ult_retraso', 'Originación'),
    'pub_rec':                  ('num_derogatorios',            'Originación'),
    'pub_rec_bankruptcies':     ('num_quiebras',                'Originación'),
    'chargeoff_within_12_mths': ('num_cancelaciones_12meses',   'Originación'),
    # --- Objetivo: solo se conocen DESPUÉS de conceder el préstamo -------
    'loan_status':              ('estado',                      'Objetivo (post-concesión)'),
    'total_rec_prncp':          ('imp_amortizado',              'Objetivo (post-concesión)'),
    'recoveries':               ('imp_recuperado',              'Objetivo (post-concesión)'),
    # --- Auxiliares: solo para filtrar o partir (no son variables del modelo)
    'issue_d':                  ('fecha_concesion',             'Auxiliar'),
    'application_type':         ('tipo_solicitud',              'Auxiliar'),
}

# Comprobación: todas las columnas elegidas existen en el archivo
faltan = set(COLUMNAS) - set(cabecera.columns)
assert not faltan, f"Columnas que no existen en el archivo: {faltan}"
print(f"Columnas seleccionadas: {len(COLUMNAS)}")

Columnas seleccionadas: 37


**Qué se hace:** se define el diccionario de columnas que se conservan: nombre original → (nombre en español, papel), y se comprueba que todas existen en el archivo.

**Qué se observa:** se seleccionan **37 columnas** y todas existen. El significado de cada una está en el diccionario de la sección 7.2.

<a id="s7-1"></a>

### 7.1 · Clasificación de las 151 columnas del archivo

In [13]:
# Reglas para clasificar las columnas descartadas
# Posteriores a la concesión (provocarían leakage): pagos, saldos, refinanciaciones, acuerdos...
POST_CONCESION_PREFIJOS = ('out_prncp', 'total_pymnt', 'total_rec_', 'last_', 'next_pymnt',
                           'hardship', 'settlement', 'debt_settlement', 'payment_plan',
                           'deferral', 'orig_projected')
POST_CONCESION = {'collection_recovery_fee', 'pymnt_plan'}
# Texto libre, enlaces y geografía
TEXTO_GEOGRAFIA = {'url', 'title', 'zip_code', 'addr_state'}
# Redundantes o propias de la operativa de la plataforma (no describen al cliente)
REDUNDANTES = {'loan_amnt', 'funded_amnt_inv', 'policy_code', 'initial_list_status', 'disbursement_method'}

def clasificar_columna(col):
    '''Devuelve (grupo, motivo) para una columna del archivo original.'''
    if col in COLUMNAS:
        return COLUMNAS[col][1], 'Seleccionada'
    if col.startswith('sec_app_') or col.endswith('_joint'):
        return 'Descartada', 'Segundo solicitante (solicitudes conjuntas, ver D8)'
    if col.startswith(POST_CONCESION_PREFIJOS) or col in POST_CONCESION:
        return 'Descartada', 'Posterior a la concesión (leakage)'
    if col in TEXTO_GEOGRAFIA:
        return 'Descartada', 'Texto libre, enlace o geografía'
    if col in REDUNDANTES:
        return 'Descartada', 'Redundante u operativa de la plataforma'
    return 'Descartada', 'Otra variable de originación (parsimonia)'

# Clasificar las 151 columnas del archivo
clasificacion = pd.DataFrame(
    [(col, *clasificar_columna(col)) for col in cabecera.columns],
    columns=['columna', 'grupo', 'motivo'],
)

# Resumen: número de columnas por grupo y motivo
clasificacion.groupby(['grupo', 'motivo']).size().to_frame('n_columnas')

n_columnas
grupo                     motivo                                                        
Auxiliar                  Seleccionada                                                 2
Descartada                Otra variable de originación (parsimonia)                   53
                          Posterior a la concesión (leakage)                          36
                          Redundante u operativa de la plataforma                      5
                          Segundo solicitante (solicitudes conjuntas, ver...          16
                          Texto libre, enlace o geografía                              4
Identificador             Seleccionada                                                 2
Objetivo (post-concesión) Seleccionada                                                 3
Originación               Seleccionada                                                30

**Qué se hace:** se clasifican las 151 columnas del archivo con unas reglas (prefijos y listas) y se resume cuántas hay de cada grupo y motivo.

**Qué se observa:** 30 de originación, 3 de objetivo, 2 auxiliares y 2 identificadores. Entre las descartadas, **36 lo son por *leakage*** (información posterior a la concesión), 53 por parsimonia, 16 por ser del segundo solicitante, 5 redundantes y 4 de texto o geografía.

In [14]:
# Tabla completa (una fila por columna) para revisar cada caso
clasificacion

,columna,grupo,motivo
0,id,Identificador,Seleccionada
1,member_id,Identificador,Seleccionada
2,loan_amnt,Descartada,Redundante u operativa de la plataforma
3,funded_amnt,Originación,Seleccionada
4,funded_amnt_inv,Descartada,Redundante u operativa de la plataforma
5,term,Originación,Seleccionada
6,int_rate,Originación,Seleccionada
7,installment,Originación,Seleccionada
8,grade,Originación,Seleccionada
9,sub_grade,Originación,Seleccionada


**Qué se hace:** se muestra la clasificación completa: una fila por columna del archivo, con su grupo y el motivo.

**Qué se observa:** sirve para revisar caso a caso que ninguna columna con *leakage* se haya colado entre las seleccionadas.

<a id="s7-2"></a>

### 7.2 · Diccionario de las variables seleccionadas

| Nombre en español | Original (Lending Club) | Papel | Significado |
|---|---|---|---|
| `id_cliente` | `id` | Identificador | Identificador único del **préstamo** en Lending Club (a pesar del nombre, identifica el préstamo). |
| `id_prestamo` | `member_id` | Identificador | Identificador del cliente. Está vacío al 100 % (anonimizado) y se elimina en D9. |
| `empleo` | `emp_title` | Originación | Puesto de trabajo que declara el cliente (texto libre). |
| `antigüedad_empleo` | `emp_length` | Originación | Años en el empleo actual, de `< 1 year` a `10+ years`. |
| `ingresos` | `annual_inc` | Originación | Ingresos anuales declarados por el cliente ($). |
| `ingresos_verificados` | `verification_status` | Originación | Si Lending Club comprobó los ingresos: `Verified`, `Source Verified` (se verificó la fuente) o `Not Verified`. |
| `vivienda` | `home_ownership` | Originación | Situación de la vivienda: `RENT` (alquiler), `MORTGAGE` (propiedad con hipoteca), `OWN` (propiedad sin hipoteca) u otras poco frecuentes. |
| `dti` | `dti` | Originación | *Debt-to-income*: pagos mensuales de todas las deudas (sin la hipoteca ni este préstamo) / ingresos mensuales, en %. |
| `descripcion` | `desc` | Originación | Texto libre del cliente explicando para qué quiere el préstamo. Se elimina en D9 (> 80 % de nulos). |
| `finalidad` | `purpose` | Originación | Finalidad declarada: consolidar deudas, pagar tarjetas, reformas, coche… |
| `principal` | `funded_amnt` | Originación | Importe concedido ($). |
| `num_cuotas` | `term` | Originación | Plazo: 36 o 60 cuotas mensuales. |
| `tipo_interes` | `int_rate` | Originación | Tipo de interés anual (%) que asigna Lending Club según el riesgo. |
| `imp_cuota` | `installment` | Originación | Cuota mensual que paga el cliente ($). |
| `rating` | `grade` | Originación | Calificación de riesgo de Lending Club: de `A` (menor riesgo) a `G` (mayor riesgo). |
| `subrating` | `sub_grade` | Originación | Subnivel dentro del rating, de `A1` a `G5` (5 niveles por rating). |
| `fico_min` / `fico_max` | `fico_range_low` / `fico_range_high` | Originación | Límites inferior y superior del rango de **puntuación FICO** del cliente (de 300 a 850; más alta = mejor historial). |
| `fecha_primer_credito` | `earliest_cr_line` | Originación | Fecha de la primera línea de crédito del cliente: indica la antigüedad de su historial. |
| `num_lineas_credito` | `open_acc` | Originación | Nº de líneas de crédito **abiertas** (tarjetas, préstamos…). |
| `num_lineas_credito_total` | `total_acc` | Originación | Nº **total** de líneas de crédito del historial, abiertas y cerradas. |
| `num_hipotecas` | `mort_acc` | Originación | Nº de préstamos hipotecarios en el informe de crédito. |
| `saldo_revolving` | `revol_bal` | Originación | Deuda pendiente en crédito *revolving* ($): crédito que se usa, se devuelve y se vuelve a usar hasta un límite (tarjetas, líneas de crédito). |
| `porc_uso_revolving` | `revol_util` | Originación | % del límite *revolving* total que está usando (saldo / límite). |
| `porc_uso_tarjetas` | `bc_util` | Originación | % del límite de sus **tarjetas bancarias** que está usando. |
| `porc_tarjetas_75p` | `percent_bc_gt_75` | Originación | % de sus tarjetas bancarias que superan el 75 % de su límite. |
| `num_consultas_6meses` | `inq_last_6mths` | Originación | Nº de consultas a su historial de crédito en los últimos 6 meses (cada nueva solicitud de crédito genera una). |
| `num_retrasos_2anios` | `delinq_2yrs` | Originación | Nº de retrasos de más de 30 días en algún pago en los últimos 2 años. |
| `num_meses_desde_ult_retraso` | `mths_since_last_delinq` | Originación | Meses desde su último retraso en un pago (vacío si no tiene ninguno registrado). |
| `num_derogatorios` | `pub_rec` | Originación | Nº de registros públicos negativos (sentencias, embargos, quiebras…). |
| `num_quiebras` | `pub_rec_bankruptcies` | Originación | Nº de quiebras en registros públicos. |
| `num_cancelaciones_12meses` | `chargeoff_within_12_mths` | Originación | Nº de deudas que un acreedor dio por perdidas en los últimos 12 meses. |
| `estado` | `loan_status` | Objetivo | Estado final del préstamo (ver sección 5). |
| `imp_amortizado` | `total_rec_prncp` | Objetivo | Principal devuelto por el cliente ($). |
| `imp_recuperado` | `recoveries` | Objetivo | Importe recuperado después del impago ($). |
| `fecha_concesion` | `issue_d` | Auxiliar | Mes de concesión del préstamo. |
| `tipo_solicitud` | `application_type` | Auxiliar | Solicitud individual o conjunta. Se usa en D8 y después se elimina. |

> **Decisión D6 · Conservar 37 columnas: 2 identificadores, 30 variables de originación, 3 para construir los objetivos y 2 auxiliares.**
> - **Las 30 variables de originación** cubren los tres bloques clásicos del *scoring* de crédito, todos conocidos al solicitar el préstamo:
>   - **Capacidad de pago:** ingresos (y si están verificados), DTI (deuda / ingresos), cuota, empleo y vivienda.
>   - **Condiciones del préstamo:** principal, plazo, tipo de interés, finalidad, rating y subrating (5 niveles dentro de cada rating).
>   - **Historial del buró de crédito:** puntuación FICO, antigüedad del historial (a partir de la fecha del primer crédito), líneas de crédito abiertas, uso de tarjetas y crédito *revolving*, consultas recientes, retrasos, registros públicos negativos y quiebras.
> - **`principal` = `funded_amnt`** (importe realmente prestado), que es la base correcta para calcular la exposición (EAD). `loan_amnt` es el importe *solicitado* y se descarta por redundante.
> - **Por qué no más:** cada variable del modelo hay que pedirla en el formulario de la demo, así que se busca un conjunto pequeño e interpretable. El resto de variables de originación puede explorarse en una versión futura.
> - **Geografía fuera** (`addr_state`, `zip_code`): añadiría decenas de categorías y puede actuar como sustituto de características protegidas (riesgo de discriminación).
> - Las 3 columnas de **objetivo nunca se usarán como variables de entrada**.

<a id="s7-3"></a>

### 7.3 · Segunda lectura: columnas seleccionadas, solo de los préstamos seleccionados

In [15]:
def es_seleccionado(bloque):
    '''Máscara con los préstamos que superaron D3, D4 y D5.'''
    # pd.to_numeric(errors='coerce') convierte en NaN el texto de las filas de totales
    return pd.to_numeric(bloque['id'], errors='coerce').isin(ids_seleccionados)

# Leer solo las columnas de D6 y solo las filas seleccionadas
prestamos = leer_por_bloques(list(COLUMNAS), filtro=es_seleccionado)
prestamos['id'] = prestamos['id'].astype('int64')

print(f"Préstamos: {len(prestamos):,} · Columnas: {prestamos.shape[1]}")
print(f"Memoria ocupada: {prestamos.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

Préstamos: 676,302 · Columnas: 37
Memoria ocupada: 721 MB


**Qué se hace:** segunda lectura por bloques: solo las 37 columnas de D6 y solo los préstamos seleccionados.

**Qué se observa:** quedan **676.302 préstamos × 37 columnas**, que ocupan unos 721 MB en memoria.

<a id="s8"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">8 · D7 · AÑO DE INICIO</span>

</div>

In [16]:
# Año de concesión de cada préstamo seleccionado
anio = pd.to_datetime(prestamos['issue_d'], format='%b-%Y').dt.year

# % de nulos por año en las variables de originación (solo las que tienen algún nulo)
variables_originacion = [lc for lc, (_, grupo) in COLUMNAS.items() if grupo.startswith('Originación')]
nulos_por_anio = prestamos[variables_originacion].isna().groupby(anio).mean() * 100
nulos_por_anio = nulos_por_anio.loc[:, nulos_por_anio.max() > 0]

# Traspuesta: una fila por variable y una columna por año
nulos_por_anio.T.round(1)

issue_d,2007,2008,2009,2010,2011,2012,2013,2014,2015
emp_title,9.000,4.100,5.600,5.900,6.600,5.900,6.400,6.500,6.700
emp_length,0.000,0.000,0.000,2.800,3.500,3.600,4.400,5.900,6.700
annual_inc,0.700,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
dti,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
desc,2.500,0.000,3.000,32.900,41.400,38.600,63.900,93.100,100.000
earliest_cr_line,4.800,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
open_acc,4.800,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
total_acc,4.800,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
mort_acc,100.000,100.000,100.000,100.000,100.000,14.000,0.000,0.000,0.000
revol_util,5.300,0.500,0.300,0.200,0.000,0.100,0.100,0.000,0.000


**Qué se hace:** % de nulos por año de concesión de cada variable de originación (solo las que tienen algún nulo). Cada fila es una variable y cada columna, un año.

**Qué se observa:** `mort_acc`, `bc_util` y `percent_bc_gt_75` están vacías al **100 % de 2007 a 2011** (Lending Club aún no las recogía), tienen ~15 % de nulos en 2012 (enero–marzo, ver el notebook 02) y ~1 % o menos desde 2013. `desc` (descripción) se deja de rellenar con los años y `mths_since_last_delinq` tiene ~50 % de nulos todos los años (tiene significado: se trata en el notebook 02).

> **Decisión D7 · Conservar solo los préstamos concedidos desde 2012.**
> - **Por qué:** `mort_acc`, `percent_bc_gt_75` y `bc_util` están vacías al 100 % entre 2007 y 2011, porque Lending Club todavía no las recogía. Es un nulo **estructural**: imputarlo sería inventar el valor durante años enteros, y el modelo podría aprender "préstamo antiguo = nulo" en lugar del riesgo real.
> - **Coste:** se pierde en torno a un 6 % de los préstamos (ver abajo). Además desaparecen los préstamos *"Does not meet the credit policy"*, todos de 2007–2010 y concedidos con una política de riesgo distinta.
> - **Alternativa descartada:** conservar 2007–2011 y eliminar esas tres variables. Se perderían variables útiles para todos los años posteriores.

In [17]:
# Préstamos que se pierden al empezar en ANIO_INICIO
antes = anio < ANIO_INICIO
print(f"Préstamos anteriores a {ANIO_INICIO}: {antes.sum():,} ({antes.mean():.1%})")

# Año de concesión de los préstamos "Does not meet the credit policy"
no_cumplen = prestamos['loan_status'].str.startswith('Does not meet')
print("\nPréstamos 'Does not meet the credit policy' por año:")
print(anio[no_cumplen].value_counts().sort_index().to_string())

# Aplicar el filtro
prestamos = prestamos[~antes]
embudo[f'5. Concedidos desde {ANIO_INICIO} (D7)'] = len(prestamos)
print(f"\nPréstamos restantes: {len(prestamos):,}")

Préstamos anteriores a 2012: 42,535 (6.3%)

Préstamos 'Does not meet the credit policy' por año:
issue_d
2007     352
2008     831
2009     565
2010    1001

Préstamos restantes: 633,767


**Qué se hace:** se cuentan los préstamos anteriores a 2012 y los *"Does not meet the credit policy"* por año, y se aplica el filtro de D7.

**Qué se observa:** se pierden **42.535 préstamos (6,3 %)**. Los *"Does not meet the credit policy"* son todos de 2007–2010, así que desaparecen con este filtro. Quedan **633.767**.

<a id="s9"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">9 · D8 · TIPO DE SOLICITUD</span>

</div>

**Variable analizada: `application_type`** · tipo de solicitud: `Individual` (un solo solicitante) o `Joint App` (solicitud conjunta de dos personas, cuyos ingresos y deudas se valoran juntos).

In [18]:
# Número de préstamos por tipo de solicitud
prestamos['application_type'].value_counts()

application_type
Individual    633528
Joint App        239
Name: count, dtype: int64

**Qué se hace:** se cuentan los préstamos por tipo de solicitud.

**Qué se observa:** casi todas son individuales (**633.528**); solo **239** son conjuntas.

> **Decisión D8 · Conservar solo las solicitudes individuales.**
> - **Por qué:** en una solicitud conjunta el riesgo depende de dos personas, y los ingresos y el DTI relevantes son los combinados (`annual_inc_joint`, `dti_joint`), que no se usan. Son muy pocas (ver tabla), así que eliminarlas apenas afecta al volumen y mantiene el mismo significado para cada variable.

In [19]:
# Filtrar y eliminar la columna auxiliar (a partir de aquí todas son individuales)
prestamos = prestamos[prestamos['application_type'] == 'Individual'].drop(columns='application_type')
embudo['6. Solo solicitudes individuales (D8)'] = len(prestamos)
print(f"Préstamos restantes: {len(prestamos):,}")

Préstamos restantes: 633,528


**Qué se hace:** se conservan solo las solicitudes individuales y se elimina la columna `application_type`, que ya no aporta nada.

**Qué se observa:** quedan **633.528 préstamos**.

<a id="s10"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">10 · D9 · COLUMNAS CASI VACÍAS</span>

</div>

In [20]:
# % de nulos de cada columna tras los filtros (solo las que tienen algún nulo)
pct_nulos = prestamos.isna().mean().sort_values(ascending=False) * 100
pct_nulos[pct_nulos > 0].round(2).to_frame('% nulos')

,% nulos
member_id,100.000
desc,85.350
mths_since_last_delinq,50.960
emp_title,6.520
emp_length,5.750
bc_util,2.210
percent_bc_gt_75,2.200
mort_acc,1.180
revol_util,0.050


**Qué se hace:** % de nulos de cada columna tras los filtros anteriores (solo las que tienen alguno).

**Qué se observa:** `member_id` está vacía al 100 % y `desc` al 85 %. El resto queda por debajo del 51 %; los nulos de esas variables se analizan en el notebook 02.

> **Decisión D9 · Eliminar las columnas con más de un 80 % de nulos**: `member_id` (100 % vacía, porque Lending Club la anonimizó) y `desc` (texto libre que casi nadie rellenaba).
> - `mths_since_last_delinq` (alrededor de la mitad de nulos) **se conserva**: el nulo tiene significado (el cliente nunca se ha retrasado) y se tratará en el notebook 02.

In [21]:
# Columnas que superan el umbral de nulos
columnas_vacias = pct_nulos[pct_nulos > UMBRAL_NULOS * 100].index.tolist()
print("Columnas eliminadas:", columnas_vacias)
prestamos = prestamos.drop(columns=columnas_vacias)

# Comprobación: cada préstamo aparece una sola vez
assert prestamos['id'].is_unique, "Hay identificadores duplicados"
print("✅ Sin identificadores duplicados")

Columnas eliminadas: ['member_id', 'desc']
✅ Sin identificadores duplicados


**Qué se hace:** se eliminan las columnas que superan el 80 % de nulos y se comprueba que ningún préstamo está repetido.

**Qué se observa:** se eliminan `member_id` y `desc`, y no hay identificadores duplicados.

<a id="s11"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">11 · D10 · NOMBRES EN ESPAÑOL</span>

</div>

> **Decisión D10 · Renombrar las columnas al español.**
> - **Por qué:** coherencia con el resto del proyecto. Los notebooks 02–08 y la API desplegada en AWS (que recibe estos nombres desde el formulario de la demo) usan estos mismos nombres. Además, las columnas se ordenan por su papel (identificador, originación, objetivos, fechas) para que la tabla sea más fácil de leer.
> - **Alternativa descartada:** mantener los nombres originales en inglés, lo que obligaría a traducirlos en cada paso hasta la API.

In [22]:
# Renombrar con el diccionario de D6 (pandas ignora las columnas que ya no existen)
prestamos = prestamos.rename(columns={lc: es for lc, (es, _) in COLUMNAS.items()})

# Ordenar las columnas como en el diccionario de D6 (agrupadas por su papel)
orden = [es for es, _ in COLUMNAS.values() if es in prestamos.columns]
prestamos = prestamos[orden]
print(list(prestamos.columns))

['id_cliente', 'empleo', 'antigüedad_empleo', 'ingresos', 'ingresos_verificados', 'vivienda', 'dti', 'finalidad', 'principal', 'num_cuotas', 'tipo_interes', 'imp_cuota', 'rating', 'subrating', 'fico_min', 'fico_max', 'fecha_primer_credito', 'num_lineas_credito', 'num_lineas_credito_total', 'num_hipotecas', 'saldo_revolving', 'porc_uso_revolving', 'porc_uso_tarjetas', 'porc_tarjetas_75p', 'num_consultas_6meses', 'num_retrasos_2anios', 'num_meses_desde_ult_retraso', 'num_derogatorios', 'num_quiebras', 'num_cancelaciones_12meses', 'estado', 'imp_amortizado', 'imp_recuperado', 'fecha_concesion']


**Qué se hace:** se renombran las columnas al español con el diccionario de D6 y se ordenan por su papel.

**Qué se observa:** quedan **34 columnas**: identificador, 29 de originación, 3 de objetivo y la fecha de concesión.

<a id="s12"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">12 · D11 · TIPOS DE DATOS</span>

</div>

> **Decisión D11 · No limpiar valores en este notebook; solo convertir las dos fechas.**
> - **Por qué:** cada notebook tiene una única responsabilidad. Textos como `' 36 months'` o `'10+ years'`, los nulos y los valores anómalos se tratan en el notebook 02 de calidad de datos. Las fechas sí se convierten aquí porque se usan para filtrar y para calcular nuevas variables más adelante.

In [23]:
# Convertir las fechas (formato 'Mes-Año', p. ej. 'Dec-2015') a tipo fecha
for col in ['fecha_concesion', 'fecha_primer_credito']:
    prestamos[col] = pd.to_datetime(prestamos[col], format='%b-%Y')

# Tipo resultante de cada columna
prestamos.dtypes.to_frame('tipo')

,tipo
id_cliente,int64
empleo,str
antigüedad_empleo,str
ingresos,float64
ingresos_verificados,str
vivienda,str
dti,float64
finalidad,str
principal,float64
num_cuotas,str


**Qué se hace:** se convierten las dos fechas a tipo fecha y se muestra el tipo de dato de cada columna.

**Qué se observa:** las fechas ya son `datetime64`. Algunas variables siguen siendo texto (`str`), como `num_cuotas` (`" 36 months"`) o `antigüedad_empleo` (`"10+ years"`); se tratarán en el notebook 02.

<a id="s13"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">13 · D12 · PARTICIÓN ENTRENAMIENTO / VALIDACIÓN</span>

</div>

> **Decisión D12 · Partición aleatoria 70 % entrenamiento / 30 % validación, estratificada por impago y con semilla fija (`random_state=42`).**
> - **Estratificar** garantiza la misma tasa de impago en los dos conjuntos.
> - **La semilla fija** hace que cualquiera que ejecute el notebook obtenga exactamente la misma partición.
> - **Alternativa considerada:** validación *out-of-time* (entrenar con los años antiguos y validar con 2015). Es más realista, pero los préstamos a 60 meses solo llegan hasta 2013 (D5), así que la validación tendría únicamente préstamos a 36 meses y no sería comparable. Puede plantearse como análisis adicional.

In [24]:
# Indicador de impago, solo para estratificar (el objetivo definitivo se crea en el notebook 04)
impago = prestamos['estado'].isin(ESTADOS_IMPAGO)

# Partición estratificada y reproducible
trabajo, val = train_test_split(
    prestamos,
    test_size=PROP_VALIDACION,   # 30 % para validación
    stratify=impago,             # misma proporción de impagos en ambos conjuntos
    random_state=SEMILLA,        # misma partición en cada ejecución
)

# Comprobación: ningún préstamo está en los dos conjuntos
assert not set(trabajo['id_cliente']) & set(val['id_cliente']), "Hay préstamos en ambos conjuntos"

# Tamaño y tasa de impago de cada conjunto
pd.DataFrame({
    'n_prestamos': [len(trabajo), len(val)],
    'tasa_impago': [trabajo['estado'].isin(ESTADOS_IMPAGO).mean(),
                    val['estado'].isin(ESTADOS_IMPAGO).mean()],
}, index=['entrenamiento', 'validación'])

,n_prestamos,tasa_impago
entrenamiento,443469,0.148
validación,190059,0.148


**Qué se hace:** se parte el conjunto en entrenamiento (70 %) y validación (30 %), estratificando por impago y con semilla fija. Se comprueba que ningún préstamo está en los dos conjuntos.

**Qué se observa:** **443.469** préstamos de entrenamiento y **190.059** de validación, con la **misma tasa de impago (14,8 %)** en ambos gracias a la estratificación.

<a id="s14"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">14 · GUARDAR LOS CONJUNTOS</span>

</div>

In [25]:
# Crear las carpetas de destino si no existen
RUTA_ENTRENAMIENTO.parent.mkdir(parents=True, exist_ok=True)
RUTA_VALIDACION.parent.mkdir(parents=True, exist_ok=True)

# Guardar en formato pickle (conserva los tipos de pandas; es lo que leen los notebooks siguientes)
trabajo.to_pickle(RUTA_ENTRENAMIENTO)
val.to_pickle(RUTA_VALIDACION)
print(f"✅ Entrenamiento: {RUTA_ENTRENAMIENTO} ({len(trabajo):,} filas)")
print(f"✅ Validación:    {RUTA_VALIDACION} ({len(val):,} filas)")

✅ Entrenamiento: ..\..\02_datos\03_Entrenamiento\train.pkl (443,469 filas)
✅ Validación:    ..\..\02_datos\02_Validacion\validacion.pkl (190,059 filas)


**Qué se hace:** se guardan los dos conjuntos en formato *pickle*, que conserva los tipos de datos de pandas.

**Qué se observa:** los archivos `train.pkl` y `validacion.pkl` quedan en `02_datos`, listos para el notebook 02.

<a id="s15"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">15 · RESUMEN: EMBUDO DE FILAS</span>

</div>

In [26]:
# Filas que quedan tras cada decisión
embudo['7. Entrenamiento (70 %)'] = len(trabajo)
embudo['8. Validación (30 %)'] = len(val)

tabla_embudo = pd.Series(embudo).to_frame('n_filas')
tabla_embudo['% sobre el original'] = tabla_embudo['n_filas'] / embudo['1. Archivo original'] * 100
tabla_embudo

,n_filas,% sobre el original
1. Archivo original,2260701,100.000
2. Sin filas de totales (D3),2260668,99.999
3. Solo préstamos terminados (D4),1348059,59.630
4. En cohortes con ≥ 95 % terminados (D5),676302,29.916
5. Concedidos desde 2012 (D7),633767,28.034
6. Solo solicitudes individuales (D8),633528,28.024
7. Entrenamiento (70 %),443469,19.616
8. Validación (30 %),190059,8.407


**Qué se hace:** tabla con las filas que quedan tras cada decisión y su % sobre el archivo original.

**Qué se observa:** las mayores pérdidas vienen de D4 (préstamos en curso, del 100 % al 59,6 %) y D5 (cohortes incompletas, hasta el 29,9 %). Al final se usa el **28 % del archivo** (633.528 préstamos), repartido en entrenamiento (19,6 %) y validación (8,4 %).

<a id="s16"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">16 · PRÓXIMOS PASOS</span>

</div>

- **02 · Calidad de datos:** convertir a número los textos (`num_cuotas`, `antigüedad_empleo`), tratar los nulos con significado (`num_meses_desde_ult_retraso`: el cliente nunca se ha retrasado) y revisar valores anómalos.
- **03 · EDA:** relación de cada variable con el impago.
- **04 · Transformación:** calcular la antigüedad del historial crediticio (`fecha_concesion` − `fecha_primer_credito`) y construir los tres objetivos a partir de `estado`, `imp_amortizado` e `imp_recuperado`.
- **05–07 · Modelos:** PD (clasificación), EAD y LGD (regresión).
- **08 · Preproducción:** las variables del modelo final serán las que se pidan en el formulario de la demo.